In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# 定义mcp
client = MultiServerMCPClient({
    "time": {
      "transport": "stdio",
      "command": "uvx",
      "args": [
        # macOS Intel 上 cryptography 50.x 没有预编译 wheel，编译会失败，这里锁定到有 wheel 的版本
        "--with", "cryptography<47",
        "mcp-server-time",
        "--local-timezone=Asia/Shanghai"
      ]
    }
})

# 用client拉取tool，注意MultiServerMCPClient是异步的，需要await
tools = await client.get_tools()


In [9]:
for tool in tools:
  print("-------------")
  print(tool.name)
  print(tool.description)
  print("-------------")

-------------
get_current_time
Get current time in a specific timezone
-------------
-------------
convert_time
Convert time between timezones
-------------
-------------
get_time_ago
Get the time a given number of days, hours, minutes and seconds ago in a specific timezone
-------------


In [11]:
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

agent = create_agent(
  model="deepseek-flash",
  tools=tools
)

response = await agent.ainvoke(
  {"messages": "现在几点了"}
)
for message in response["messages"]:
  message.pretty_print()

================================ Human Message =================================

现在几点了
================================== Ai Message ==================================
Tool Calls:
  get_current_time (call_00_kgTlhT9YOQycAiFcNr913091)
 Call ID: call_00_kgTlhT9YOQycAiFcNr913091
  Args:
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-11-18T13:34:15+08:00",\n  "day_of_week": "Wednesday",\n  "is_dst": false\n}', 'id': 'lc_7e997701-bfc6-4715-a5c5-46c7a3b8e082'}]
================================== Ai Message ==================================

现在是 **2026年11月18日（周三）13:34**，北京时间（UTC+8）。

如果你所在时区不同，告诉我，我帮你换算～


/var/folders/m4/zhtdd1k17kg00_vw288ynmvr0000gn/T/ipykernel_26846/1038992727.py:11: RuntimeWarning: coroutine 'Pregel.ainvoke' was never awaited
  response = await agent.ainvoke(


In [13]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
  "kiwi-com-flight-search": {
    "transport": "http",
    "url": "https://mcp.kiwi.com"
  },
  "time": {
    "transport": "stdio",
    "command": "uvx",
    "args": [
      # macOS Intel 上 cryptography 50.x 没有预编译 wheel，编译会失败，这里锁定到有 wheel 的版本
      "--with", "cryptography<47",
      "mcp-server-time",
      "--local-timezone=Asia/Shanghai"
    ]
  }
})

tools = await client.get_tools()


In [14]:
agent = create_agent(
  model="deepseek-flash",
  tools=tools,
  system_prompt=(
    "你是一名旅行agent。帮助用户找到最好的航班。"
    "不要追问用户问题，如果用户使用中文，则使用zh-cn作为区域设置值"
  )
)

In [16]:
response = await agent.ainvoke(
  {"messages": "查一下下周一从成都去北京的航班"}
)
for message in response["messages"]:
  message.pretty_print()

================================ Human Message =================================

查一下下周一从成都去北京的航班
================================== Ai Message ==================================

我先确认一下今天的日期，然后为您搜索。
Tool Calls:
  get_current_time (call_00_WbrEnL8c0ksY2ALwvVCo7933)
 Call ID: call_00_WbrEnL8c0ksY2ALwvVCo7933
  Args:
    timezone: Asia/Shanghai
================================= Tool Message =================================
Name: get_current_time

[{'type': 'text', 'text': '{\n  "timezone": "Asia/Shanghai",\n  "datetime": "2026-11-18T13:49:42+08:00",\n  "day_of_week": "Wednesday",\n  "is_dst": false\n}', 'id': 'lc_7e10acbe-dc50-456d-83ba-c2cfcda28b23'}]
================================== Ai Message ==================================

今天是2026年11月18日（周三），下周一就是**2026年11月23日**。我来搜索成都 → 北京的航班。
Tool Calls:
  search-flight (call_00_tSDPuIsuZCGo0OKt4k1Q7231)
 Call ID: call_00_tSDPuIsuZCGo0OKt4k1Q7231
  Args:
    flyFrom: 成都
    flyTo: 北京
    departureDate: 23/11/2026
    locale: zh-cn
    curren